# Earnings Call NLP — Exploratory Data Analysis

**Dataset:** 18,755 earnings call transcripts, 2,876 tickers, 2017–2023
**Source:** [Motley Fool Scraped Earnings Call Transcripts](https://www.kaggle.com/datasets/tpotterer/motley-fool-scraped-earnings-call-transcripts) via Kaggle

This notebook covers the raw dataset structure, transcript length, section composition,
and speaker role distribution — the foundation for understanding what the pipeline is working with.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

sys.path.insert(0, "../src")
import config

# ── Consistent color palette ──────────────────────────────────────────────────
C = dict(
    positive="#2E86AB",   # steel blue  — positive sentiment, top quartile
    negative="#E84855",   # coral red   — negative sentiment, bottom quartile
    neutral="#6C757D",    # slate gray  — neutral / mid-range
    accent="#F4A261",     # amber       — highlights, overlap comparison
    grid="#E9ECEF",       # light gray  — gridlines
)

FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

def save_fig(fig, name, width=900, height=500):
    """Save interactive plotly figure as high-res PNG and display."""
    path = FIGURES_DIR / f"{name}.png"
    fig.write_image(str(path), width=width, height=height, scale=2)
    fig.show()
    print(f"Saved → {path}")

import data_ingestion

raw = data_ingestion.load_transcripts(config.RAW_DIR / "motley-fool-data.pkl")
raw["word_count"] = raw["transcript_text"].str.split().str.len()
raw["year"] = raw["date"].dt.year


## Dataset Overview

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Metric": [
        "Total transcripts", "Unique tickers", "Date range",
        "Exchanges covered", "Mean words / transcript",
        "Median words / transcript", "Has explicit Q&A section",
    ],
    "Value": [
        f"{len(raw):,}",
        f"{raw['ticker'].nunique():,}",
        f"{raw['date'].min().date()} → {raw['date'].max().date()}",
        f"{raw['exchange'].nunique():,}",
        f"{raw['word_count'].mean():,.0f}",
        f"{raw['word_count'].median():,.0f}",
        f"{raw['has_qa_section'].sum():,} ({100*raw['has_qa_section'].mean():.1f}%)",
    ],
}).set_index("Metric")
display(summary)

# ── Call volume by fiscal quarter ─────────────────────────────────────────────
qtr_counts = raw.groupby("fiscal_quarter").size().reset_index(name="count")
qtr_counts = qtr_counts.sort_values("fiscal_quarter")

fig = go.Figure(go.Bar(
    x=qtr_counts["fiscal_quarter"],
    y=qtr_counts["count"],
    marker_color=C["positive"],
    hovertemplate="%{x}: %{y:,} calls<extra></extra>",
))
fig.update_layout(
    title="Earnings Calls per Fiscal Quarter",
    xaxis_title="Fiscal Quarter", yaxis_title="Transcript Count",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(tickangle=45, gridcolor=C["grid"], tickfont=dict(size=10)),
    yaxis=dict(gridcolor=C["grid"]),
    height=420,
)
save_fig(fig, "eda_call_volume")


## Transcript Length Distribution

In [ ]:
median_wc = raw["word_count"].median()
p25, p75 = raw["word_count"].quantile(0.25), raw["word_count"].quantile(0.75)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=raw["word_count"], nbinsx=80,
    marker_color=C["positive"], opacity=0.85,
    hovertemplate="~%{x:,} words: %{y:,} transcripts<extra></extra>",
))
fig.add_vline(x=median_wc, line_dash="dash", line_color=C["negative"], line_width=2,
              annotation_text=f"Median: {median_wc:,.0f}", annotation_position="top right",
              annotation=dict(font_size=12, font_color=C["negative"]))
fig.update_layout(
    title="Transcript Length Distribution",
    xaxis_title="Word Count", yaxis_title="Number of Transcripts",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor=C["grid"]),
    yaxis=dict(gridcolor=C["grid"]),
    height=420, showlegend=False,
)
save_fig(fig, "eda_word_count_dist")

print(f"Word count — mean: {raw['word_count'].mean():,.0f} | "
      f"median: {median_wc:,.0f} | "
      f"IQR: [{p25:,.0f}, {p75:,.0f}] | "
      f"max: {raw['word_count'].max():,}")


## Section Breakdown: Prepared Remarks vs Q&A

~54% of transcripts contain an explicit `"Questions and Answers:"` header.
The remaining ~46% are split using a heuristic: the first analyst speaker line
(`"Name -- Firm -- Analyst"`) after a minimum number of prepared-remarks lines.
This is a key pipeline decision — see `src/preprocessing.py` for details.


In [ ]:
# Load only the columns needed to avoid pulling 457MB into memory
scores_meta = pd.read_parquet(
    "../data/cache/finbert_scores_nooverlap.parquet",
    columns=["ticker", "date", "source_section", "chunk_index"],
)

# Chunk count per transcript per section
section_counts = (
    scores_meta.groupby(["ticker", "date", "source_section"])
    .size()
    .reset_index(name="chunk_count")
)

pr = section_counts.loc[section_counts["source_section"] == "prepared_remarks", "chunk_count"]
qa = section_counts.loc[section_counts["source_section"] == "qa_session", "chunk_count"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Prepared Remarks", "Q&A Session"],
)
fig.add_trace(go.Histogram(x=pr, nbinsx=50, marker_color=C["positive"],
                            opacity=0.85, name="Prepared Remarks"), row=1, col=1)
fig.add_trace(go.Histogram(x=qa, nbinsx=50, marker_color=C["accent"],
                            opacity=0.85, name="Q&A Session"), row=1, col=2)

for col in [1, 2]:
    fig.update_xaxes(title_text="Chunks per Transcript", gridcolor=C["grid"], row=1, col=col)
    fig.update_yaxes(title_text="Transcripts", gridcolor=C["grid"], row=1, col=col)

fig.update_layout(
    title="Section Length Distribution (chunks per transcript)",
    plot_bgcolor="white", paper_bgcolor="white",
    showlegend=False, height=420,
)
save_fig(fig, "eda_section_breakdown", width=1000)

print(f"Prepared remarks — median chunks: {pr.median():.0f}, mean: {pr.mean():.1f}")
print(f"Q&A sessions     — median chunks: {qa.median():.0f}, mean: {qa.mean():.1f} "
      f"(across {len(qa):,} transcripts with Q&A)")


## Speaker Role Distribution

In [ ]:
role_counts = (
    scores_full["role"].value_counts().reset_index()
)
role_counts.columns = ["role", "chunks"]
role_counts = role_counts.sort_values("chunks", ascending=True)

ROLE_LABELS = {
    "ceo": "CEO", "cfo": "CFO", "analyst": "Analyst",
    "operator": "Operator", "ir": "Investor Relations",
    "executive": "Other Executive", "unknown": "Unknown",
}
ROLE_COLORS = {
    "CEO": C["positive"], "CFO": C["positive"],
    "Analyst": C["negative"], "Investor Relations": C["accent"],
    "Operator": C["neutral"], "Other Executive": C["positive"], "Unknown": C["neutral"],
}

role_counts["label"] = role_counts["role"].map(ROLE_LABELS).fillna(role_counts["role"])
colors = role_counts["label"].map(ROLE_COLORS).fillna(C["neutral"])

fig = go.Figure(go.Bar(
    x=role_counts["chunks"],
    y=role_counts["label"],
    orientation="h",
    marker_color=list(colors),
    text=role_counts["chunks"].apply(lambda x: f"{x/1e3:.0f}k"),
    textposition="outside",
    hovertemplate="%{y}: %{x:,} chunks<extra></extra>",
))
fig.update_layout(
    title="Total Chunks by Speaker Role",
    xaxis_title="Chunks (1.25M total)", yaxis_title="",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor=C["grid"]),
    height=400, showlegend=False,
    margin=dict(l=160),
)
save_fig(fig, "eda_speaker_roles")


## Sample Transcript — Clean Parsing

In [ ]:
# Show one transcript with explicit Q&A and one heuristic-detected
explicit = raw[raw["has_qa_section"] == True].iloc[3]
heuristic = raw[raw["has_qa_section"] == False].iloc[3]

for label, row in [("Explicit Q&A header (has_qa_section=True)", explicit),
                    ("No explicit header — heuristic split (has_qa_section=False)", heuristic)]:
    print(f"{'='*70}")
    print(f"{label}")
    print(f"Ticker: {row['ticker']}  |  Date: {row['date'].date()}  |  Quarter: {row['fiscal_quarter']}")
    print(f"{'─'*70}")
    print(row["transcript_text"][:700])
    print("  [...]\n")
